[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/webpug/geospatial/blob/main/olmo_earth_embeddings_demo.ipynb)

# OLMoEarth · a neighborhood in learned features

Turn real Sentinel-2 observations into spatial features with an openly available pretrained model. You will read a neighborhood in RGB, explore its embedding colors and unsupervised groups, ask which cells resemble a query, and compare your own named places. **Features are not labels:** similarity suggests a pattern to investigate, not a semantic conclusion.

**Route:** private copy → setup → controls → connect → pretrained weights → real imagery → explore → compare places.

Start with the four public Seattle-area examples, **2024**, **64 × 64 pixels** (640 m square), and **V1 Nano**. A GPU is recommended; CPU works more slowly. V1 Nano is a lightweight, stable V1 default, **not the newest family or a maximum-accuracy choice**. Tiny and Base are optional and use more memory.

## Your private workspace

This is a **public template**. Before entering personal names or coordinates, choose **File → Save a copy in Drive**. Open **Share**, set **General access → Restricted**, and inspect/remove collaborators. A launch link cannot enforce privacy. Never save personal entries, outputs, credentials or widget state back to GitHub; do not use **Save a copy in GitHub** for personal work.

Inputs and results stay in runtime memory. Earth Engine receives coordinates when you explicitly run acquisition; the optional Maps Search sends a query and key to Google only when clicked. Model weights download from Hugging Face and may be cached by its client. We do not write your places or imagery to Drive, a bucket or a repository. Results and charts can still expose your entries. `omit_outputs=true` and cleared template outputs are conveniences, not privacy guarantees: clear outputs, restart/delete the runtime, and inspect the file before sharing. Closing a browser alone does not clear runtime memory.

Edits invalidate in-memory results. Existing plots remain labeled snapshots until rerun. There are no substitute datasets, random-weight fallbacks or precomputed demonstration outputs.

## 1 · Install the small inference package

Requires **Python ≥3.11**, PyTorch ≥2.7, an Earth Engine account and your own Cloud project with Earth Engine enabled. Colab normally supplies PyTorch; the installer retains a compatible installation. Choose **Runtime → Change runtime type → GPU** if available. If an already-imported dependency changes, restart the runtime and run from here again.

The minimal package is pinned to the reviewed commit below. Its dependencies resolve at installation time. First weight loading needs internet; a failed download stops the lab.

In [ ]:
#@title Install dependencies (run once per fresh runtime)
import sys
if sys.version_info < (3, 11):
    raise RuntimeError("Use a Python 3.11+ Colab runtime.")
%pip install -q "olmoearth-pretrain-minimal @ git+https://github.com/allenai/olmoearth_pretrain_minimal.git@96eeae43032d50dfc637d1200819b3966994392b" earthengine-api pyproj numpy pandas scipy scikit-learn matplotlib ipywidgets requests
from google.colab import output
output.enable_custom_widget_manager()

## 2 · Load the self-contained lab code

These embedded helpers are part of this notebook: no live repository helper or second notebook is imported. They implement explicit grids, validity masks, pretrained inference and local controls. Run all four folded cells once.

In [ ]:
# Embedded source: notebook_sources/earth_helpers.py
"""Shared implementation embedded verbatim in both standalone public notebooks.

No downloads or authentication occur on import. See scripts/build_earth_notebooks.py.
"""
import json
import math
import time
from dataclasses import dataclass, asdict
from datetime import date
import numpy as np
from pyproj import Transformer

CODE_SHA = '96eeae43032d50dfc637d1200819b3966994392b'
S2_COLLECTION = 'COPERNICUS/S2_SR_HARMONIZED'
ALPHA_COLLECTION = 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
MODEL_BANDS = ('B02', 'B03', 'B04', 'B08', 'B05', 'B06', 'B07', 'B8A', 'B11', 'B12', 'B01', 'B09')
EE_BANDS = ('B2', 'B3', 'B4', 'B8', 'B5', 'B6', 'B7', 'B8A', 'B11', 'B12', 'B1', 'B9')
ALPHA_BANDS = tuple(f'A{i:02d}' for i in range(64))
PATCH = 4
SCALE = 10
MAX_PLACES = 8
DEFAULT_PLACES = {
    'Seattle urban': (47.6062, -122.3321),
    'Discovery Park': (47.6597, -122.4165),
    'Seattle port': (47.5806, -122.3509),
    'Tiger Mountain forest': (47.4881, -121.9460),
}


def query_coordinates(latitude, longitude):
    if isinstance(latitude, bool) or isinstance(longitude, bool):
        raise ValueError('Coordinates must be numbers, not booleans.')
    try:
        lat, lon = float(latitude), float(longitude)
    except (TypeError, ValueError, OverflowError):
        raise ValueError('Enter numeric latitude and longitude.') from None
    if not (math.isfinite(lat) and math.isfinite(lon)):
        raise ValueError('Coordinates must be finite.')
    if not (-80 <= lat <= 84 and -180 <= lon <= 180):
        raise ValueError('UTM needs latitude -80 to 84 and longitude -180 to 180.')
    return lat, lon


def validate_places(places, minimum=2):
    if not minimum <= len(places) <= MAX_PLACES:
        raise ValueError(f'Use {minimum} to {MAX_PLACES} places.')
    for name, coords in places.items():
        if not isinstance(name, str) or not name.strip() or len(name) > 100:
            raise ValueError('Place names need 1–100 characters.')
        query_coordinates(*coords)
    return tuple((name, *query_coordinates(*coords)) for name, coords in places.items())


@dataclass(frozen=True)
class Settings:
    project: str
    year: int = 2024
    size: int = 64
    model: str = 'OLMOEARTH_V1_NANO'

    def __post_init__(self):
        if not isinstance(self.project, str) or not self.project.strip():
            raise ValueError('Enter your own Earth Engine project ID; rerun authentication.')
        object.__setattr__(self, 'project', self.project.strip())
        if type(self.year) is not int or not 2017 <= self.year <= 2100:
            raise ValueError('Use a calendar year from 2017 to 2100; coverage is checked live.')
        if type(self.size) is not int or self.size not in (32, 64, 128):
            raise ValueError('Chip size must be 32, 64 or 128 pixels.')
        if self.model not in ('OLMOEARTH_V1_NANO', 'OLMOEARTH_V1_TINY', 'OLMOEARTH_V1_BASE'):
            raise ValueError('Choose a supported pretrained V1 model.')


def quarters(year):
    # Representative composite dates, deliberately NOT claimed as scene dates.
    return [dict(start=f'{year}-{month:02d}-01',
                 end=f'{year + 1}-01-01' if month == 10 else f'{year}-{month + 3:02d}-01',
                 representative=f'{year}-{month + 1:02d}-15')
            for month in (1, 4, 7, 10)]


def timestamps(year):
    dates = [date.fromisoformat(q['representative']) for q in quarters(year)]
    return np.array([[[d.day, d.month - 1, d.year] for d in dates]], dtype=np.int64)


def make_grid(lat, lon, size):
    lat, lon = query_coordinates(lat, lon)
    if type(size) is not int or size not in (32, 64, 128):
        raise ValueError('Chip size must be 32, 64 or 128.')
    zone = min(60, int((lon + 180) // 6) + 1)
    # Standard UTM exceptions for Norway / Svalbard.
    if 56 <= lat < 64 and 3 <= lon < 12:
        zone = 32
    if 72 <= lat < 84 and 0 <= lon < 42:
        zone = 31 if lon < 9 else 33 if lon < 21 else 35 if lon < 33 else 37
    crs = f'EPSG:{(32600 if lat >= 0 else 32700) + zone}'
    x, y = Transformer.from_crs('EPSG:4326', crs, always_xy=True).transform(lon, lat)
    return dict(crsCode=crs, dimensions=dict(width=size, height=size),
                affineTransform=dict(scaleX=SCALE, shearX=0, translateX=x-size*SCALE/2,
                                     shearY=0, scaleY=-SCALE, translateY=y+size*SCALE/2))


def affine_list(grid):
    a = grid['affineTransform']
    return [a[k] for k in ('scaleX', 'shearX', 'translateX', 'shearY', 'scaleY', 'translateY')]


def grid_region(ee, grid):
    a = grid['affineTransform']
    x, y = a['translateX'], a['translateY']
    return ee.Geometry.Rectangle([x, y-grid['dimensions']['height']*SCALE,
                                  x+grid['dimensions']['width']*SCALE, y],
                                 proj=grid['crsCode'], geodesic=False)


def read_pixels(ee, image, bands, grid):
    """Export validity explicitly; filling transports pixels, never creates validity."""
    bands = list(bands)
    valid_names = ['valid_' + b for b in bands]
    selected = image.select(bands)
    validity = selected.mask().rename(valid_names).unmask(0, sameFootprint=False)
    packed = selected.unmask(0, sameFootprint=False).addBands(validity).toFloat()
    response = ee.data.computePixels(dict(expression=packed, fileFormat='NUMPY_NDARRAY',
                                          bandIds=bands + valid_names, grid=grid))
    expected = (grid['dimensions']['height'], grid['dimensions']['width'])
    if response.shape != expected or not set(bands + valid_names) <= set(response.dtype.names or ()):
        raise ValueError('Earth Engine returned an unexpected pixel grid or band schema.')
    values = np.stack([response[b] for b in bands], axis=-1).astype(np.float32)
    valid = np.stack([response[b] > 0 for b in valid_names], axis=-1) & np.isfinite(values)
    return np.where(valid, values, np.nan), valid


def acquire_s2(ee, settings, grid):
    region = grid_region(ee, grid)
    values, masks, counts = [], [], []
    for window in quarters(settings.year):  # Sequential, four bounded requests.
        collection = (ee.ImageCollection(S2_COLLECTION).filterBounds(region)
                      .filterDate(window['start'], window['end']))
        count = collection.size().getInfo()
        if not count:
            raise ValueError(f"No Sentinel-2 scenes in {window['start']} to {window['end']}.")
        def prepare(image):
            scl = image.select('SCL')
            good = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7))
            # Nearest-neighbor resampling to the common grid before the median.
            # Native 20/60 m bands remain lower-information observations.
            return (image.select(list(EE_BANDS)).updateMask(good)
                    .reproject(crs=grid['crsCode'], crsTransform=affine_list(grid)))
        composite = collection.map(prepare).median()
        data, valid = read_pixels(ee, composite, EE_BANDS, grid)
        if not valid.any():
            raise ValueError(f"No valid Sentinel-2 pixels in quarter starting {window['start']}.")
        values.append(data)
        masks.append(valid)
        counts.append(int(count))
    return np.stack(values, axis=2), np.stack(masks, axis=2), counts


def acquire_alpha(ee, settings, grid):
    collection = (ee.ImageCollection(ALPHA_COLLECTION).filterBounds(grid_region(ee, grid))
                  .filterDate(f'{settings.year}-01-01', f'{settings.year + 1}-01-01'))
    if not collection.size().getInfo():
        raise ValueError(f'No AlphaEarth coverage for this AOI in {settings.year}. Try 2024 or another AOI.')
    data, valid = read_pixels(ee, collection.sort('system:index').mosaic(), ALPHA_BANDS, grid)
    if not valid.all(axis=-1).any():
        raise ValueError('AlphaEarth has no valid vectors on this exact grid.')
    return data, valid.all(axis=-1)


def block_view(array, patch=PATCH):
    h, w = array.shape[:2]
    if h % patch or w % patch:
        raise ValueError('Spatial dimensions must divide by patch size.')
    return array.reshape(h//patch, patch, w//patch, patch, *array.shape[2:])


def conservative_tokens(valid):
    if valid.ndim != 4 or valid.shape[2:] != (4, 12):
        raise ValueError('Expected H,W,4,12 validity.')
    return block_view(valid.all(axis=(2, 3))).all(axis=(1, 3))


def unit_vectors(values, valid):
    values = np.asarray(values, dtype=np.float32)
    norm = np.linalg.norm(np.where(np.isfinite(values), values, 0), axis=-1)
    good = np.asarray(valid, bool) & np.isfinite(values).all(axis=-1) & (norm > 1e-12)
    result = np.full(values.shape, np.nan, dtype=np.float32)
    result[good] = values[good] / norm[good, None]
    return result, good


def aggregate_alpha(values, valid):
    valid = valid & np.isfinite(values).all(axis=-1)
    counts = block_view(valid).sum(axis=(1, 3))
    sums = block_view(np.where(valid[..., None], values, 0)).sum(axis=(1, 3))
    means = sums / np.maximum(counts[..., None], 1)
    vectors, good = unit_vectors(means, counts > 0)
    return vectors, good, counts / (PATCH * PATCH)


def prepare_sample(dn, valid, year, device='cpu'):
    import torch
    from olmoearth_pretrain_minimal import Normalizer
    from olmoearth_pretrain_minimal.olmoearth_pretrain_v1.utils.constants import Modality
    from olmoearth_pretrain_minimal.olmoearth_pretrain_v1.utils.datatypes import MaskedOlmoEarthSample, MaskValue
    if dn.ndim != 4 or dn.shape[2:] != (4, 12) or dn.shape != valid.shape:
        raise ValueError('Expected matching H,W,4,12 DN and validity arrays.')
    if tuple(Modality.SENTINEL2_L2A.band_order) != MODEL_BANDS:
        raise ValueError('Installed model band order differs from this pinned lab.')
    valid = valid & np.isfinite(dn)
    # Feed raw L2A DN to the official normalizer. Do NOT divide by 10000.
    normalized = Normalizer(std_multiplier=2).normalize(Modality.SENTINEL2_L2A,
                                                        np.where(valid, dn, 0)[None])
    normalized = np.where(valid[None], normalized, 0).astype(np.float32)
    mask = np.where(valid[None], MaskValue.ONLINE_ENCODER.value, MaskValue.MISSING.value)
    return MaskedOlmoEarthSample(
        sentinel2_l2a=torch.as_tensor(normalized, device=device),
        sentinel2_l2a_mask=torch.as_tensor(mask, dtype=torch.long, device=device),
        timestamps=torch.as_tensor(timestamps(year), dtype=torch.long, device=device))


def infer_olmo(model, dn, valid, year, device):
    import torch
    good = conservative_tokens(valid & np.isfinite(dn))
    if not good.any():
        raise ValueError('No fully valid 40 m tokens across all bands and quarters. Try another AOI/year.')
    sample = prepare_sample(dn, valid, year, device)
    with torch.inference_mode():
        features = model.encoder(sample, patch_size=4, input_res=10, fast_pass=True)['tokens_and_masks'].sentinel2_l2a
        if features.ndim != 6 or tuple(features.shape[:4]) != (1, *good.shape, 4):
            raise ValueError('Unexpected encoder output; expected B,H/4,W/4,T,S,D.')
        pooled = features.mean(dim=(3, 4))[0].float().cpu().numpy()
    vectors, good = unit_vectors(pooled, good)
    if not good.any():
        raise ValueError('The encoder produced no finite, nonzero valid output vectors.')
    return vectors, good


class Pipeline:
    """Only model weights are reused in memory. Imagery/results are never cached."""
    def __init__(self):
        self.model = None
        self.model_key = None
        self.device = None
        self.load_seconds = None

    def load(self, settings):
        import torch
        from olmoearth_pretrain_minimal import ModelID, load_model_from_id
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        key = (CODE_SHA, settings.model, device)
        if self.model_key == key and self.model is not None:
            return
        self.model = self.model_key = None  # A failed replacement cannot expose old weights.
        start = time.perf_counter()
        model_id = getattr(ModelID, settings.model)
        model = load_model_from_id(model_id, load_weights=True)
        model = model.to(device).eval()
        self.model, self.model_key, self.device = model, key, device
        self.load_seconds = time.perf_counter() - start

    def run(self, ee, settings, lat, lon, comparison=False):
        if self.model is None or self.model_key[:2] != (CODE_SHA, settings.model):
            raise ValueError('Run the pretrained model load cell for the selected model first.')
        grid = make_grid(lat, lon, settings.size)
        start = time.perf_counter()
        dn, valid, counts = acquire_s2(ee, settings, grid)
        download_s2 = time.perf_counter() - start
        vectors, good = infer_olmo(self.model, dn, valid, settings.year, self.device)
        olmo_seconds = time.perf_counter() - start
        result = dict(grid=grid, dn=dn, input_valid=valid, olmo=vectors, olmo_valid=good,
                      shared_valid=good.copy(), settings=asdict(settings))
        runtime = dict(olmo_pipeline_seconds=olmo_seconds, sentinel2_download_seconds=download_s2,
                       olmo_model_load_seconds=self.load_seconds, device=self.device)
        coverage = dict(sentinel2_band_time_fraction=float(valid.mean()),
                        sentinel2_quarter_fractions=valid.mean(axis=(0, 1, 3)).tolist(),
                        olmo_token_fraction=float(good.mean()))
        if comparison:
            start = time.perf_counter()
            alpha, alpha_valid = acquire_alpha(ee, settings, grid)
            download_alpha = time.perf_counter() - start
            alpha, alpha_good, fraction = aggregate_alpha(alpha, alpha_valid)
            shared = good & alpha_good
            if not shared.any():
                raise ValueError('No common valid output cells for both models.')
            result.update(alpha=alpha, alpha_valid=alpha_good, shared_valid=shared,
                          alpha_block_coverage=fraction)
            runtime.update(alpha_pipeline_seconds=time.perf_counter()-start,
                           alpha_download_seconds=download_alpha)
            coverage.update(alpha_10m_fraction=float(alpha_valid.mean()),
                            alpha_token_fraction=float(alpha_good.mean()),
                            alpha_mean_block_fraction=float(fraction.mean()),
                            shared_token_fraction=float(shared.mean()))
        result['provenance'] = dict(
            sentinel2_collection=S2_COLLECTION, alpha_collection=ALPHA_COLLECTION if comparison else None,
            year=settings.year, quarterly_windows=quarters(settings.year), quarter_scene_counts=counts,
            timestamp_meaning='Representative composite dates, NOT actual scene acquisition dates',
            timestamps_day_zero_based_month_year=timestamps(settings.year).tolist(),
            grid=grid, scale_metres=SCALE, ee_bands=EE_BANDS, model_bands=MODEL_BANDS,
            alpha_bands=ALPHA_BANDS if comparison else None, model_id=settings.model, code_sha=CODE_SHA,
            normalization='Official Normalizer(std_multiplier=2) on raw L2A DN',
            masking='SCL 4,5,6,7; output OLMo requires every pixel/band/quarter valid',
            resampling='Nearest neighbor to explicit 10 m UTM grid before quarterly median',
            aggregation='OLMo time/bandset mean; Alpha valid-only 4x4 mean; L2 normalize final vectors',
            coverage=coverage, runtime=runtime)
        return result


def spatial_query(grid, latitude=None, longitude=None, row=None, column=None):
    if row is None or column is None:
        lat, lon = query_coordinates(latitude, longitude)
        x, y = Transformer.from_crs('EPSG:4326', grid['crsCode'], always_xy=True).transform(lon, lat)
        a = grid['affineTransform']
        column = math.floor((x-a['translateX']) / (SCALE*PATCH))
        row = math.floor((a['translateY']-y) / (SCALE*PATCH))
    if type(row) is not int or type(column) is not int:
        raise ValueError('Token row/column must be integers.')
    if not (0 <= row < grid['dimensions']['height']//PATCH and
            0 <= column < grid['dimensions']['width']//PATCH):
        raise ValueError('Query is outside this chip. Change the query or acquire a new chip.')
    return row, column


def similarity_to_query(vectors, valid, row, column):
    if not valid[row, column]:
        raise ValueError('Query token is invalid. Choose a colored cell on the coverage/PCA panel.')
    scores = np.full(valid.shape, np.nan)
    # Both operands are from ONE model. No cross-model cosine API exists here.
    scores[valid] = np.clip(vectors[valid] @ vectors[row, column], -1, 1)
    return scores


def place_vector(result, model):
    valid = result['shared_valid']
    vector, good = unit_vectors(result[model][valid].mean(axis=0)[None], np.array([True]))
    if not good[0]:
        raise ValueError('Place mean has zero norm; cannot compare it.')
    return vector[0]


def pairwise(vectors):
    # A single homogeneous model matrix; differing feature counts fail at stacking.
    values = np.stack(vectors)
    if values.ndim != 2:
        raise ValueError('Expected a single model matrix of place vectors.')
    return np.clip(values @ values.T, -1, 1)


def rank_agreement(a, b):
    from scipy.stats import spearmanr
    a, b = np.asarray(a), np.asarray(b)
    if a.shape != b.shape:
        raise ValueError('Rank inputs must describe the same neighbors.')
    if len(a) < 3 or not np.isfinite(a).all() or not np.isfinite(b).all():
        return None
    if np.ptp(a) < 1e-10 or np.ptp(b) < 1e-10:
        return None
    score = float(spearmanr(a, b).statistic)
    return score if np.isfinite(score) else None


def pca_colors(vectors, valid):
    from sklearn.decomposition import PCA
    selected = vectors[valid]
    if len(selected) < 3:
        raise ValueError('PCA needs at least three valid cells; try another AOI/year.')
    colors = PCA(n_components=3, svd_solver='full').fit_transform(selected)
    lo, hi = np.percentile(colors, [2, 98], axis=0)
    colors = np.clip((colors-lo) / np.maximum(hi-lo, 1e-8), 0, 1)
    rgba = np.zeros((*valid.shape, 4))
    rgba[valid, :3] = colors
    rgba[valid, 3] = 1
    return rgba


def rgb_image(result):
    # Quarter 3 reference image, DN stretch for DISPLAY only; never feeds inference.
    rgb = result['dn'][:, :, 2, [2, 1, 0]]
    valid = result['input_valid'][:, :, 2, [2, 1, 0]].all(axis=-1)
    rgba = np.zeros((*valid.shape, 4))
    rgba[valid, :3] = np.clip(rgb[valid] / 3000, 0, 1) ** (1/1.6)
    rgba[valid, 3] = 1
    return rgba


def paint_chip(result, clusters=5):
    import matplotlib.pyplot as plt
    from sklearn.cluster import KMeans
    models = ['olmo', 'alpha'] if 'alpha' in result else ['olmo']
    valid = result['shared_valid']
    panels = [('Sentinel-2 · Q3 RGB', rgb_image(result), None)]
    for name in models:
        panels.append((f'{name.title()} · independent PCA', pca_colors(result[name], valid), None))
    if len(models) == 1:
        selected = result['olmo'][valid]
        k = min(clusters, len(np.unique(selected, axis=0)))
        labels = np.full(valid.shape, np.nan)
        labels[valid] = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(selected)
        panels.append((f'Unsupervised groups · k={k}', labels, 'tab10'))
    panels.append(('Common valid cells' if 'alpha' in result else 'Valid output cells', valid.astype(float), 'gray'))
    fig, axes = plt.subplots(1, len(panels), figsize=(5*len(panels), 5), constrained_layout=True)
    width = result['grid']['dimensions']['width']*SCALE
    for ax, (title, values, cmap) in zip(axes, panels):
        ax.set_facecolor('#cbd5e1')
        ax.imshow(values, origin='upper', extent=[0, width, width, 0], cmap=cmap, interpolation='nearest')
        ax.set(title=title, xlabel='Metres east from west edge', ylabel='Metres south from north edge')
    fig.suptitle(f"{result['settings']['year']} · {result['grid']['crsCode']} · north up · gray/transparent = missing")
    plt.show()


def paint_query(result, row, column):
    import matplotlib.pyplot as plt
    models = ['olmo', 'alpha'] if 'alpha' in result else ['olmo']
    fig, axes = plt.subplots(1, len(models), figsize=(6*len(models), 5), squeeze=False, constrained_layout=True)
    for ax, model in zip(axes.flat, models):
        scores = similarity_to_query(result[model], result['shared_valid'], row, column)
        im = ax.imshow(scores, vmin=-1, vmax=1, cmap='viridis', interpolation='nearest')
        ax.set_facecolor('#cbd5e1')
        ax.scatter([column], [row], c='white', edgecolors='black', marker='*', s=160)
        ax.set(title=f'{model.title()} · cosine to query', xlabel='40 m token column', ylabel='40 m token row')
        fig.colorbar(im, ax=ax, label='Within-model cosine')
    plt.show()


def compare_places(results, query_name):
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    names = list(results)
    if query_name not in results:
        raise ValueError('Choose one of the current places as ranking query.')
    models = ['olmo', 'alpha'] if 'alpha' in results[names[0]] else ['olmo']
    matrices = {m: pairwise([place_vector(results[n], m) for n in names]) for m in models}
    count = len(models) + (len(models) == 2)
    fig, axes = plt.subplots(1, count, figsize=(6*count, 5), squeeze=False, constrained_layout=True)
    for ax, model in zip(axes.flat, models):
        im = ax.imshow(matrices[model], vmin=-1, vmax=1, cmap='viridis')
        ax.set_title(f'{model.title()} · place cosine')
        fig.colorbar(im, ax=ax)
    if len(models) == 2:
        im = axes.flat[-1].imshow(matrices['olmo']-matrices['alpha'], vmin=-2, vmax=2, cmap='coolwarm')
        axes.flat[-1].set_title('OLMo − Alpha similarity\nDiagnostic, not accuracy')
        fig.colorbar(im, ax=axes.flat[-1])
    for ax in axes.flat:
        ax.set_xticks(range(len(names)), names, rotation=40, ha='right')
        ax.set_yticks(range(len(names)), names)
        # User names are literal text, including dollar signs and backslashes.
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_parse_math(False)
    plt.show()
    q = names.index(query_name)
    others = [i for i in range(len(names)) if i != q]
    table = pd.DataFrame({'place': [names[i] for i in others]})
    for model in models:
        table[model + ' cosine'] = matrices[model][q, others]
        table[model + ' rank'] = table[model + ' cosine'].rank(ascending=False, method='average')
    display(table.sort_values(models[0] + ' rank'))
    if len(models) == 2:
        agreement = rank_agreement(matrices['olmo'][q, others], matrices['alpha'][q, others])
        print('Neighbor Spearman:', 'undefined (need ≥3 nonconstant neighbor scores)' if agreement is None else f'{agreement:.3f}')
    return matrices

In [ ]:
# Embedded source: notebook_sources/lab_ui.py
"""Local controls; Earth Engine is called only by the explicitly run acquisition cells."""
from html import escape
import ipywidgets as widgets
from IPython.display import display, clear_output


class PlacesEditor:
    def __init__(self, places, changed=lambda: None):
        self.places, self.changed = places, changed
        self.name = widgets.Text(description='Name:')
        self.latitude = widgets.Text(description='Latitude:')
        self.longitude = widgets.Text(description='Longitude:')
        self.selection = widgets.Dropdown(description='Saved place:')
        self.add = widgets.Button(description='Add / update', button_style='primary')
        self.remove = widgets.Button(description='Remove selected')
        self.reset = widgets.Button(description='Reset defaults')
        self.status, self.summary = widgets.HTML(), widgets.HTML()
        self.selection.observe(self._select, names='value')
        self.add.on_click(self._add)
        self.remove.on_click(self._remove)
        self.reset.on_click(self._reset)
        self.view = widgets.VBox([self.selection, self.name, self.latitude, self.longitude,
                                 widgets.HBox([self.add, self.remove, self.reset]), self.status, self.summary])
        self._refresh()

    def _refresh(self, selected=None):
        self.selection.options = [('Choose a place…', None)] + [(n, n) for n in self.places]
        self.selection.value = selected
        self.remove.disabled = selected is None
        self.summary.value = f'<b>{len(self.places)} / {MAX_PLACES} places</b><ul>' + ''.join(
            f'<li>{escape(n)}: {lat}, {lon}</li>' for n, (lat, lon) in self.places.items()) + '</ul>'

    def _select(self, change):
        name = change['new']
        self.remove.disabled = name is None
        if name is not None:
            lat, lon = self.places[name]
            self.name.value, self.latitude.value, self.longitude.value = name, str(lat), str(lon)

    def _add(self, _):
        try:
            name = self.name.value.strip()
            coords = query_coordinates(self.latitude.value, self.longitude.value)
            candidate = {**self.places, name: coords}
            validate_places(candidate, minimum=1)
        except ValueError as exc:
            self.status.value = escape(str(exc))
            return
        self.places[name] = coords
        self.changed()
        self._refresh(name)
        self.status.value = escape(f'Saved {name}. Run acquisition to update results.')

    def _remove(self, _):
        name = self.selection.value
        if name is not None:
            del self.places[name]
            self.changed()
            self._refresh()
            self.status.value = escape(f'Removed {name}. Run acquisition to update results.')

    def _reset(self, _):
        self.places.clear()
        self.places.update(DEFAULT_PLACES)
        self.changed()
        self._refresh()
        self.status.value = 'Restored four public examples. Run acquisition to update results.'


class LabControls:
    def __init__(self, places, comparison=False):
        self.places, self.comparison = places, comparison
        self.pipeline = Pipeline()
        self.connected_project = None
        self.chip = self.place_results = None
        self.chip_key = self.places_key = None
        self.project = widgets.Text(description='EE project:', placeholder='Your own project ID (required)')
        self.year = widgets.IntText(value=2024, description='Year:')
        self.size = widgets.Dropdown(options=[32, 64, 128], value=64, description='Chip pixels:')
        self.model = widgets.Dropdown(options=[('V1 Nano · lightweight', 'OLMOEARTH_V1_NANO'),
                                              ('V1 Tiny', 'OLMOEARTH_V1_TINY'),
                                              ('V1 Base · more memory', 'OLMOEARTH_V1_BASE')],
                                      description='Pretrained:')
        self.latitude = widgets.Text(value='47.6062', description='Chip latitude:')
        self.longitude = widgets.Text(value='-122.3321', description='Chip longitude:')
        self.status = widgets.HTML(value='Edit locally, then run authentication and model cells below.')
        self.editor = PlacesEditor(places, self.invalidate)
        self.use_place = widgets.Button(description='Use selected as chip', layout=widgets.Layout(width='auto'))
        self.use_place.on_click(self._use_place)
        for widget in (self.project, self.year, self.size, self.model, self.latitude, self.longitude):
            widget.observe(self.invalidate, names='value')
        self.view = widgets.VBox([
            widgets.HTML('<h3>Explore a 640 m Seattle neighborhood</h3><p>64 × 64 inputs → 16 × 16 output cells. Edits stay local.</p>'),
            self.project, widgets.HBox([self.year, self.size]), self.model,
            widgets.HBox([self.latitude, self.longitude]), self.status,
            widgets.HTML('<h4>PLACES · compare up to eight neighborhoods</h4>'), self.editor.view, self.use_place])

    def invalidate(self, *_):
        self.chip = self.place_results = None
        self.chip_key = self.places_key = None
        self.status.value = 'Inputs changed. Previous plots are snapshots; rerun acquisition before using results.'

    def _use_place(self, _):
        name = self.editor.selection.value
        if name is not None:
            self.set_applied(*self.places[name])

    def set_applied(self, lat, lon):
        lat, lon = query_coordinates(lat, lon)
        self.latitude.value, self.longitude.value = str(lat), str(lon)

    def settings(self):
        return Settings(self.project.value, self.year.value, self.size.value, self.model.value)

    def snapshot(self):
        return (self.settings(), query_coordinates(self.latitude.value, self.longitude.value),
                tuple((n, *coords) for n, coords in self.places.items()), self.comparison)

    def authenticate(self, ee):
        self.invalidate()
        self.connected_project = None
        settings = self.settings()  # Blank fails before auth.
        ee.Authenticate()
        ee.Initialize(project=settings.project)
        self.connected_project = settings.project
        self.status.value = 'Connected. Run the pretrained model load cell next.'

    def load_model(self):
        self.invalidate()
        self.pipeline.load(self.settings())
        print(f'Loaded real pretrained {self.settings().model} on {self.pipeline.device}.')
        print(f'Model loading: {self.pipeline.load_seconds:.1f} s (reported separately from data pipelines).')

    def _ready(self):
        settings = self.settings()
        if self.connected_project != settings.project:
            raise ValueError('Run authentication with this project ID first.')
        return settings

    def acquire_chip(self, ee):
        self.chip = self.chip_key = None
        settings = self._ready()
        key = self.snapshot()
        result = self.pipeline.run(ee, settings, *key[1], comparison=self.comparison)
        self.chip, self.chip_key = result, key
        self.status.value = 'Chip ready. Run the panels and local spatial query cells.'
        return result

    def acquire_places(self, ee):
        self.place_results = self.places_key = None
        settings = self._ready()
        entries = validate_places(self.places)
        key = self.snapshot()
        results = {}
        for index, (name, lat, lon) in enumerate(entries, 1):
            print(f'Acquiring place {index}/{len(entries)} …')
            results[name] = self.pipeline.run(ee, settings, lat, lon, comparison=self.comparison)
        self.place_results, self.places_key = results, key  # Publish only complete runs.
        self.status.value = 'All places ready. Choose a ranking query below.'
        return results

    def require_chip(self):
        if self.chip is None or self.chip_key != self.snapshot():
            raise ValueError('No current chip. Run acquisition for the current inputs first.')
        return self.chip

    def require_places(self):
        if self.place_results is None or self.places_key != self.snapshot():
            raise ValueError('No current place results. Run place acquisition for the current inputs first.')
        return self.place_results


def spatial_query_controls(lab):
    result = lab.require_chip()
    n = result['grid']['dimensions']['width']//PATCH
    mode = widgets.Dropdown(options=['Token row / column', 'Latitude / longitude'], description='Query by:')
    row = widgets.BoundedIntText(value=n//2, min=0, max=n-1, description='Token row:')
    column = widgets.BoundedIntText(value=n//2, min=0, max=n-1, description='Token column:')
    latitude = widgets.Text(value=lab.latitude.value, description='Latitude:')
    longitude = widgets.Text(value=lab.longitude.value, description='Longitude:')
    button = widgets.Button(description='Show similarity', button_style='primary')
    output = widgets.Output()
    def show(_):
        with output:
            clear_output(wait=True)
            try:
                current = lab.require_chip()
                if mode.value == 'Token row / column':
                    r, c = spatial_query(current['grid'], row=row.value, column=column.value)
                else:
                    r, c = spatial_query(current['grid'], latitude.value, longitude.value)
                if not current['shared_valid'][r, c]:
                    raise ValueError('Query cell is missing. Choose a valid cell from the coverage panel.')
                paint_query(current, r, c)
            except ValueError as exc:
                print(str(exc))
    button.on_click(show)
    view = widgets.VBox([mode, widgets.HBox([row, column]), widgets.HBox([latitude, longitude]), button, output])
    display(view)
    return view


def ranking_controls(lab):
    results = lab.require_places()
    query = widgets.Dropdown(options=list(results), description='Query place:')
    button = widgets.Button(description='Compare neighbors', button_style='primary')
    output = widgets.Output()
    def show(_):
        with output:
            clear_output(wait=True)
            try:
                compare_places(lab.require_places(), query.value)
            except ValueError as exc:
                print(str(exc))
    button.on_click(show)
    view = widgets.VBox([query, button, output])
    display(view)
    return view

In [ ]:
# Embedded source: notebook_sources/maps_key.py
#@title ▶️ Optional: set a runtime Google Maps API key
import getpass
import warnings


def set_runtime_maps_key():
    global _maps_runtime_api_key
    # Clear any previous key before prompting, including on cancellation or failure.
    _maps_runtime_api_key = None
    try:
        with warnings.catch_warnings():
            # Stop getpass before its fallback can read input with echo enabled.
            warnings.simplefilter("error", getpass.GetPassWarning)
            _maps_runtime_api_key = getpass.getpass(
                "GOOGLE_MAPS_API_KEY (hidden; blank clears runtime key and uses Secrets): "
            ).strip() or None
    except getpass.GetPassWarning:
        _maps_runtime_api_key = None
        print("Hidden input is unavailable. Runtime key cleared; use Colab Secrets with notebook access.")
    except (KeyboardInterrupt, EOFError):
        _maps_runtime_api_key = None
        print("Key setup cancelled. Runtime key cleared; Search will use Colab Secrets.")
    except Exception:
        _maps_runtime_api_key = None
        print("Key setup could not complete. Runtime key cleared; use Colab Secrets with notebook access.")
    else:
        if _maps_runtime_api_key:
            print("Runtime key set. Run the Maps search cell, then click Search.")
        else:
            print("Runtime key cleared. Search will use Colab Secrets with notebook access.")

In [ ]:
# Embedded source: notebook_sources/maps_search.py
#@title ▶️ Optional: search for coordinates with Google Maps
import requests



def search_place_candidates(text_query):
    # Called only by Search. The optional runtime key takes precedence over Secrets.
    api_key = globals().get("_maps_runtime_api_key")
    guidance = ("Set GOOGLE_MAPS_API_KEY in Colab Secrets with notebook access (preferred), "
                "or run the optional runtime Google Maps API key setup cell immediately above search.")
    if not isinstance(api_key, str) or not api_key.strip():
        try:
            from google.colab import userdata
            api_key = userdata.get("GOOGLE_MAPS_API_KEY")
        except Exception:
            return [], guidance
    if not isinstance(api_key, str) or not api_key.strip():
        return [], guidance
    api_key = api_key.strip()

    try:
        response = requests.post(
            "https://places.googleapis.com/v1/places:searchText",
            json={"textQuery": text_query, "pageSize": 5},
            headers={"X-Goog-Api-Key": api_key,
                     "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.location,places.attributions"},
            timeout=(5, 20), allow_redirects=False,
        )
        if response.status_code == 403:
            return [], "Search denied (403). Check billing, Places API (New), and key restrictions."
        if response.status_code == 429:
            return [], "Search quota reached (429). Wait and check your quota before retrying."
        if response.status_code >= 500:
            return [], "Google Maps is temporarily unavailable. Try again later."
        if response.status_code != 200:
            return [], "Search could not complete. Check your API setup and try again."
        try:
            payload = response.json()
        except ValueError:
            return [], "Search returned an unreadable response. Try again later."
        if not isinstance(payload, dict) or not isinstance(payload.get("places", []), list):
            return [], "Search returned an unexpected response. Try again later."

        def clean_text(value):
            # Also redact a secret if a malformed service response echoes it.
            return value.replace(api_key, "[redacted]") if isinstance(value, str) else ""

        candidates = []
        for place in payload.get("places", [])[:5]:
            if not isinstance(place, dict):
                continue
            location = place.get("location")
            if not isinstance(location, dict):
                continue
            try:
                lat, lon = query_coordinates(location.get("latitude"), location.get("longitude"))
            except ValueError:
                continue
            display_name = place.get("displayName")
            name = clean_text(display_name.get("text")) if isinstance(display_name, dict) else ""
            address = clean_text(place.get("formattedAddress"))
            attributions = []
            providers = place.get("attributions", [])
            if isinstance(providers, list):
                for provider in providers:
                    if isinstance(provider, dict):
                        attribution = " — ".join(filter(None, [clean_text(provider.get("provider")),
                                                                 clean_text(provider.get("providerUri"))]))
                        if attribution:
                            attributions.append(attribution)
            candidates.append({"name": name or address or "Unnamed place", "address": address,
                               "lat": lat, "lon": lon, "attributions": attributions})
        if not candidates:
            message = ("No results. Try a more specific place or city." if not payload.get("places")
                       else "No results with valid coordinates. Try another search or enter coordinates manually.")
            return [], message
        return candidates, "Choose a candidate, then click a transfer button."
    except requests.exceptions.Timeout:
        return [], "Search timed out. Try again later."
    except Exception:
        # Never expose upstream exception text, response bodies, headers, or the key.
        return [], "Search could not complete. Check your connection and API setup."
    finally:
        api_key = None


class PlacesSearch:
    def __init__(self):
        self.candidates = []
        self.query = widgets.Text(description="Find:", placeholder="City, address, business, or description",
                                  layout=widgets.Layout(width="95%"))
        self.search = widgets.Button(description="Search", button_style="primary")
        self.selection = widgets.Dropdown(description="Candidate:", options=[("Choose a candidate…", None)],
                                           layout=widgets.Layout(width="95%"))
        self.add = widgets.Button(description="Add selected to PLACES", layout=widgets.Layout(width="auto"))
        self.use = widgets.Button(description="Use as chip center")
        self.status = widgets.HTML()
        self.results = widgets.HTML()
        self.search.on_click(self._search)
        self.selection.observe(self._select, names="value")
        self.query.observe(self._edited, names="value")
        self.add.on_click(self._add)
        self.use.on_click(self._use)
        self.view = widgets.VBox([self.query, self.search, self.status, self.results,
                                  self.selection, widgets.HBox([self.add, self.use])])
        self._clear()

    def _clear(self):
        self.candidates = []
        self.selection.options = [("Choose a candidate…", None)]
        self.selection.value = None
        self.add.disabled = self.use.disabled = True
        self.results.value = ""

    def _edited(self, _):
        self._clear()
        self.status.value = "Click Search to look up this query."

    def _select(self, _):
        self.add.disabled = self.use.disabled = self.selection.value is None

    def _search(self, _):
        if self.search.disabled:
            return
        self._clear()
        text_query = self.query.value.strip()
        if not text_query:
            self.status.value = "Enter a place or description to search."
            return
        self.search.disabled = self.query.disabled = True
        self.status.value = "Searching…"
        try:
            candidates, message = search_place_candidates(text_query)
            self.candidates = candidates
            self.status.value = escape(message)
            rows = []
            options = [("Choose a candidate…", None)]
            for i, candidate in enumerate(candidates):
                label = (f"{candidate['name']} — {candidate['address']} "
                         f"({candidate['lat']}, {candidate['lon']})")
                options.append((label, i))
                providers = "; ".join(candidate["attributions"])
                rows.append(f"<li>{escape(label)}<br>{escape(providers)}</li>")
            if rows:
                self.results.value = (
                    '<div style="border:1px solid #ccc;padding:10px">'
                    '<span translate="no" style="font:400 14px sans-serif;white-space:nowrap;'
                    'color:#1f1f1f;background:white">Google Maps</span><ol>'
                    + "".join(rows) + "</ol></div>"
                )
            self.selection.options = options
            self.selection.value = None
        except Exception:
            self._clear()
            self.status.value = "Search could not complete. Try again later."
        finally:
            self.search.disabled = self.query.disabled = False

    def _candidate(self):
        index = self.selection.value
        if index is None or not 0 <= index < len(self.candidates):
            self.status.value = "Choose a candidate first."
            return None
        return self.candidates[index]

    def _add(self, _):
        candidate = self._candidate()
        if candidate is None:
            return
        # Resolve the current editor at click time, even after its cell is rerun.
        name = candidate["name"]
        base, suffix = name, 2
        while name in PLACES:
            name = f"{base} ({suffix})"
            suffix += 1
        places_editor.name.value = name
        places_editor.latitude.value = str(candidate["lat"])
        places_editor.longitude.value = str(candidate["lon"])
        places_editor._add(None)
        self.status.value = places_editor.status.value

    def _use(self, _):
        candidate = self._candidate()
        if candidate is None:
            return
        lab.set_applied(candidate["lat"], candidate["lon"])
        self.status.value = "Chip center applied. Run chip acquisition to download imagery."

    def close(self):
        self.search.on_click(self._search, remove=True)
        self.add.on_click(self._add, remove=True)
        self.use.on_click(self._use, remove=True)
        self.selection.unobserve(self._select, names="value")
        self.query.unobserve(self._edited, names="value")
        for widget in (self.add, self.use, *self.view.children, self.view):
            widget.close()

In [ ]:
#@title Open controls · edits make no Earth Engine requests
COMPARISON = False
if 'PLACES' not in globals():
    PLACES = DEFAULT_PLACES.copy()
if 'lab' not in globals():
    lab = LabControls(PLACES, comparison=COMPARISON)
places_editor = lab.editor
display(lab.view)

### Optional · Google Maps coordinate search

Manual entry needs no Maps key. For search, enable billing and **Places API (New)** and restrict your key to that API. Requests may incur charges. Preferred: add `GOOGLE_MAPS_API_KEY` to **Colab Secrets (key icon)** and enable notebook access. Alternatively run the masked prompt below; it never falls back to visible input. A blank entry or cancellation clears the runtime key; otherwise it takes precedence over Secrets until cleared or the runtime is deleted.

Click **Search**, select a candidate, then **Add selected to PLACES** or **Use as chip center**. These transfer buttons only edit controls; run acquisition explicitly afterward. Google Maps attribution appears with search results. This search chooses a chip center; the later spatial query stays inside the acquired chip.

In [ ]:
#@title Optional: set / clear a runtime Maps key (skip when using Secrets)
set_runtime_maps_key()

In [ ]:
#@title Optional: open Maps search (no request until Search)
if 'places_search' in globals():
    places_search.close()
places_search = PlacesSearch()
display(places_search.view)

## 3 · Connect and load real pretrained weights

Enter your own project in the blank control above, then run both cells. Blank projects fail before authentication. Changing the model requires rerunning model loading. The loader always requests pretrained weights; errors propagate rather than switching to random features.

In [ ]:
#@title Authenticate Earth Engine with your project
import ee
lab.authenticate(ee)

In [ ]:
#@title Load the selected real V1 pretrained model
# Nano uses load_model_from_id(ModelID.OLMOEARTH_V1_NANO, load_weights=True).
# Tiny/Base use the matching ModelID with the same mandatory load_weights=True.
lab.load_model()

## 4 · Acquire the selected neighborhood

Each quarter of the selected year contributes a **median Sentinel-2 SR_HARMONIZED composite**. SCL classes **4, 5, 6, 7** retain vegetation, bare surfaces, water and unclassified surfaces; clouds, shadows, snow and other classes are excluded. SCL screening is imperfect. February/May/August/November **15th** are explicitly **representative dates**, not actual scene dates. Timestamps use day, **zero-based month**, year.

The chip is centered on your coordinates in its local UTM zone, with a fixed 10 m north-up affine transform. Native 20/60 m bands are sampled by nearest neighbor; this does not create 10 m detail. Earth Engine returns a structured NumPy array plus explicit per-band validity on that exact grid. Four sequential small downloads avoid bucket exports.

Raw L2A **DN** values go directly to `Normalizer(std_multiplier=2)`—no division by 10000. The band order is **B02 B03 B04 B08 B05 B06 B07 B8A B11 B12 B01 B09**. Model inputs and masks are **B,H,W,T,12**; mask 0 means online encoder, mask 3 means missing. Transport fill values are never counted as valid data.

Encoder output is **B,H/4,W/4,T,S,D** at patch size 4 and input resolution 10 m. We average time and band-set dimensions and normalize the final vectors. A plotted OLMo token conservatively requires every input pixel, band and quarter to be valid. Its nominal footprint is **40 × 40 m**, but the contextual receptive field extends beyond that footprint. An empty collection or no valid output stops with an error.

In [ ]:
#@title Acquire the chip · explicit network + inference step
# Results are published only after all required steps succeed.
lab.acquire_chip(ee)
print(json.dumps(lab.require_chip()['provenance'], indent=2))

### Read the provenance before reading the colors

Coverage fractions describe retained input values, each quarter, output tokens and (in the comparison) shared support. The printed record includes collection, year, windows, representative timestamps, band order, affine transform, code SHA, model ID and pooling rule.

Timing is **end-to-end pipeline runtime**, not a training or inference speed contest. OLMo's pipeline includes Earth Engine download, preparation and local inference; Alpha's includes download and aggregation of an already-computed product. Model loading is reported separately. Hardware, network and service caches affect the numbers. Imagery is downloaded again on every acquisition; there is no data cache to hide changed inputs.

## 5 · See the neighborhood

All panels share the same spatial footprint and north-up orientation. RGB is a stretched **third-quarter** reference composite. PCA compresses features into three display colors fitted on valid cells; these are **not RGB, land-cover labels or absolute feature meanings**. Missing data is gray/transparent. PCA needs at least three valid cells.

K-means groups similar embedding vectors without labels. Group numbers and colors have no fixed semantics, and changing the chip or number of clusters can change them. Check patterns against RGB and local knowledge before forming a hypothesis.

In [ ]:
#@title Draw aligned panels from the current chip
paint_chip(lab.require_chip())

## 6 · Ask a spatial question

Use a **zero-based token row/column** (0…15 for a 64 px chip), or enter latitude/longitude inside the chip. Click **Show similarity** to draw a cosine heatmap. The star marks the query footprint; gray cells are excluded. The center cell may be missing—choose a colored valid cell instead.

A high score means the model represents two cells similarly, not that they share a known label. These controls run locally on acquired features and never contact Earth Engine.

In [ ]:
#@title Interactive local query
query_view = spatial_query_controls(lab)

## 7 · Which neighborhoods resemble one another?

Edit **PLACES** above: add/update, select/remove, or reset four public examples (urban, park, port, forest). Select a place and click **Use selected as chip** to explore it in the earlier cells. Use **2–8 places** for this comparison; requests run sequentially. Each place is a neighborhood mean over its valid output cells, normalized after averaging—not a single point measurement.

Run acquisition after editing, then choose a named ranking query. This downloads a new chip per place. If any place fails, no partial result or previous ranking is reused. Coverage differences can affect neighborhood summaries.

In [ ]:
#@title Acquire all PLACES · sequential network + inference step
lab.acquire_places(ee)
# Full provenance per place is retained; printed records can contain your coordinates.
for name, result in lab.require_places().items():
    print(name)
    print(json.dumps(result['provenance'], indent=2))

In [ ]:
#@title Compare place similarities and choose a ranking query locally
ranking_view = ranking_controls(lab)

### Interpret the comparison carefully

Matrix entries are within-model cosine between pooled neighborhood vectors. The ranking excludes the query itself. Urban, park, port and forest are names for public example locations, not training labels or predictions. Similarity may be affected by season, cloud filtering, water coverage, neighborhood composition and the model's spatial context.

**Try next:** change one coordinate or the year, rerun acquisition and inspect both coverage and patterns. Keep the inputs and aggregation fixed when comparing runs. If you see no data, try another place/year; if you run out of memory, choose Nano and 32 px. If authentication fails, check project registration/access and rerun the connection cell. This lab has offline contract tests; live authenticated Earth Engine and hosted Colab execution must still be verified in your runtime.

## Credits, sources and licenses

- **Ai2 OLMoEarth team**: [official project](https://github.com/allenai/olmoearth_pretrain), [platform](https://olmoearth.allenai.org/), [inference quickstart](https://github.com/allenai/olmoearth_pretrain/blob/main/docs/Inference-Quickstart.md), [minimal package](https://github.com/allenai/olmoearth_pretrain_minimal), and [V1 Nano pretrained weights](https://huggingface.co/allenai/OlmoEarth-v1-Nano). Model artifacts use the [OLMoEarth Artifact License](https://github.com/allenai/olmoearth_pretrain/blob/main/LICENSE); this is **not a blanket MIT license for weights**.
- **Ivan Zvonkov**: inspiration from the official [Earth Engine embedding generation notebook](https://github.com/allenai/olmoearth_pretrain/blob/main/scripts/tools/generate_embeddings.ipynb). This small-chip lab uses the minimal package and direct pixel requests instead of that notebook's bucket workflow.
- **Google / Google DeepMind AlphaEarth**: [Satellite Embedding V1 dataset, usage and terms](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_SATELLITE_EMBEDDING_V1_ANNUAL). Sentinel-2: [harmonized L2A catalog and terms](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED); pixel downloads follow [computePixels](https://developers.google.com/earth-engine/apidocs/ee-data-computepixels).
- **Teodora Szasz, PhD**: the adapted lab experience builds on her original **“You Are Hand-Labelling Data Google Already Described for Free”**, preserved in [this repository](https://github.com/webpug/geospatial/blob/main/external_inspirations/standout_systems_labelling_lab.ipynb). [Her work](https://teodora.coach/) inspired our [AlphaEarth playground](https://colab.research.google.com/github/webpug/geospatial/blob/main/alpha_earth_embeddings_demo.ipynb) and its place-editor and privacy patterns; the optional Maps helpers here are adapted from that demo.

[All labs and requirements](https://github.com/webpug/geospatial#launch-the-labs) · [OLMoEarth demo](https://colab.research.google.com/github/webpug/geospatial/blob/main/olmo_earth_embeddings_demo.ipynb) · [OLMoEarth / AlphaEarth comparison](https://colab.research.google.com/github/webpug/geospatial/blob/main/olmo_vs_alpha_earth_comparison.ipynb)